# Unidad 5 · Evaluación del agente de inspección

El cuaderno contrasta la trayectoria **del turno** con la esperada y verifica el contenido de la respuesta final por separado. Configura `GOOGLE_API_KEY` en el archivo local `.env` y ejecuta `taller.cmd evaluar`.

**Versión corregida.** Las instrucciones de inspección exigen dígito, confianza y
decisión juntos. El validador de respuestas conserva las mismas expresiones regulares
y tolerancias que la versión anterior. La celda de regresión comprueba los casos de rechazo.
La ausencia de clave produce estado BLOQUEADA, nunca una aprobación implícita.

In [1]:
from pathlib import Path
import os
import sys

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "modelo.py").is_file())
os.chdir(raiz)
sys.path.insert(0, str(raiz))
os.environ["KERAS_BACKEND"] = "jax"
os.environ.setdefault("JAX_PLATFORMS", "cpu")

Out[1]: 'cpu'


## Referencia independiente

La etiqueta del dataset permite inspeccionar el error de la CNN. Para evaluar al agente, la referencia es la lectura que realmente recibió de la CNN.

In [2]:
import json
import re
import unicodedata
from uuid import uuid4

import cv2
import pandas as pd
from IPython.display import display
from src.agente import consultar, decision, UMBRAL, MODELO_PREDETERMINADO
from src.modelo import predecir

imagen = cv2.imread("assets/caso_prueba.png", cv2.IMREAD_GRAYSCALE)
caso_u4 = json.loads(Path("assets/caso_prueba.json").read_text(encoding="utf-8"))
referencia = predecir(imagen)
assert referencia["digito"] == caso_u4["digito"]
assert abs(referencia["confianza_pct"] - caso_u4["confianza_pct"]) <= 0.02
accion = decision(referencia["confianza_pct"])
print("Caso U4:", caso_u4)
print("Salida independiente de la CNN:", referencia)
print(f"Umbral operativo: {UMBRAL:.2f} %. Acción de referencia: {accion}.")
print("Modelo configurado:", os.getenv("GEMINI_MODEL") or MODELO_PREDETERMINADO)


Caso U4: {'dataset': 'sklearn.load_digits', 'indice': 1149, 'etiqueta_real': 8, 'digito': 2, 'confianza_pct': 57.95, 'variante': 'Dropout'}
Salida independiente de la CNN: {'digito': 2, 'confianza_pct': 57.95}
Umbral operativo: 90.00 %. Acción de referencia: REVISAR.
Modelo configurado: gemini-3.5-flash-lite


## Casos y criterios

Saludo y recuerdo comparten hilo. Las dos inspecciones usan hilos nuevos. Para la política se contrasta también el argumento de la segunda herramienta con la salida de la primera.

In [3]:
lote = "Aliso"
hilo_saludo = str(uuid4())
casos = [
    ("Saludo", f"Hola, mi lote se llama {lote}. Salúdame.", [], None, hilo_saludo),
    ("Lectura", "Lee el dígito de la casilla e indica su confianza.", ["leer_digito"], imagen, str(uuid4())),
    ("Inspección", "Lee el dígito, indica su confianza y decide si aceptar o revisar la lectura.",
     ["leer_digito", "aplicar_regla"], imagen, str(uuid4())),
    ("Memoria", "¿Qué nombre te indiqué para mi lote?", [], None, hilo_saludo),
]
for nombre, pregunta, esperadas, _, _ in casos:
    print(nombre, "|", pregunta, "| trayectoria esperada:", esperadas)

def normalizar(texto):
    limpio = unicodedata.normalize("NFKD", texto.casefold())
    return "".join(ch for ch in limpio if not unicodedata.combining(ch)).replace("*", "")

def resultado_correcto(nombre, respuesta):
    frase = normalizar(respuesta)
    if nombre == "Saludo":
        return bool(re.search(r"\b(hola|saludos|buen[oa]s?|bienvenid[oa])\b", frase))
    if nombre == "Memoria":
        return bool(re.search(r"\baliso\b", frase))
    digito = re.search(r"digito\s*(?:(?:es|predicho)\s*)?[:=]?\s*(\d)\b", frase)
    porcentaje = re.search(r"(\d+(?:[.,]\d+)?)\s*%", frase)
    lectura_ok = bool(digito and int(digito[1]) == referencia["digito"] and porcentaje
                      and abs(float(porcentaje[1].replace(",", ".")) - referencia["confianza_pct"]) <= 0.05)
    if nombre == "Lectura":
        return lectura_ok
    decision_final = re.search(r"decision\s*[:=]\s*(aceptar|revisar)\b", frase)
    return lectura_ok and bool(decision_final and decision_final[1] == accion.casefold())


Saludo | Hola, mi lote se llama Aliso. Salúdame. | trayectoria esperada: []
Lectura | Lee el dígito de la casilla e indica su confianza. | trayectoria esperada: ['leer_digito']
Inspección | Lee el dígito, indica su confianza y decide si aceptar o revisar la lectura. | trayectoria esperada: ['leer_digito', 'aplicar_regla']
Memoria | ¿Qué nombre te indiqué para mi lote? | trayectoria esperada: []


## Regresión del criterio de aceptación (sin API)

Estos ejemplos comprueban el validador, no son respuestas generadas por Gemini.
La respuesta incompleta debe seguir fallando, igual que un porcentaje, dígito o
decisión incorrectos.

In [4]:
correcta = f"Dígito: {referencia['digito']}. Confianza: {referencia['confianza_pct']}%. Decisión: {accion}"
assert not resultado_correcto("Inspección", "Decisión: REVISAR")
assert resultado_correcto("Inspección", correcta)
assert not resultado_correcto("Inspección", f"Dígito: {(referencia['digito'] + 1) % 10}. Confianza: {referencia['confianza_pct']}%. Decisión: {accion}")
assert not resultado_correcto("Inspección", f"Dígito: {referencia['digito']}. Confianza: 1%. Decisión: {accion}")
otra_accion = "ACEPTAR" if accion == "REVISAR" else "REVISAR"
assert not resultado_correcto("Inspección", f"Dígito: {referencia['digito']}. Confianza: {referencia['confianza_pct']}%. Decisión: {otra_accion}")
print("5 comprobaciones del validador superadas. No equivalen a una ejecución con Gemini.")


5 comprobaciones del validador superadas. No equivalen a una ejecución con Gemini.


## Ejecución con Google AI Studio

El historial se obtiene del checkpointer. `consultar` lee los `tool_calls` posteriores al identificador del mensaje actual. La columna de resultado exige los datos del caso según el formato del agente. Una frase equivalente con otro formato puede necesitar revisión manual.

In [5]:
filas = []
if not os.getenv("GOOGLE_API_KEY"):
    print("BLOQUEADA: GOOGLE_API_KEY no está configurada. No se ejecutaron los cuatro casos con Gemini.")
else:
    for nombre, pregunta, esperadas, adjunto, hilo in casos:
        try:
            respuesta, llamadas = consultar(pregunta, adjunto, hilo)
        except Exception as exc:
            print(f"{nombre}: consulta interrumpida ({type(exc).__name__}). Comprueba modelo, conexión y cuota.")
            break
        reales = [llamada["nombre"] for llamada in llamadas]
        argumento_ok = None
        if nombre == "Inspección":
            argumentos = [c["argumentos"].get("confianza_pct") for c in llamadas if c["nombre"] == "aplicar_regla"]
            argumento_ok = (len(argumentos) == 1 and isinstance(argumentos[0], (int, float))
                            and abs(argumentos[0] - referencia["confianza_pct"]) <= 0.05)
        filas.append({"caso": nombre, "esperadas": esperadas, "invocadas": reales,
                      "trayectoria_ok": reales == esperadas, "confianza_transmitida_ok": argumento_ok,
                      "resultado_ok": resultado_correcto(nombre, respuesta), "respuesta": respuesta})
        print(nombre, "| llamadas reales:", llamadas, "| respuesta:", respuesta)
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(filas))
    print("Casos con respuesta en vivo:", len(filas), "de", len(casos))
    if filas:
        print("Trayectorias correctas:", sum(f["trayectoria_ok"] for f in filas))
        print("Resultados correctos:", sum(f["resultado_ok"] for f in filas))


from datetime import datetime, timezone
estado = ("BLOQUEADA_SIN_API_KEY" if not os.getenv("GOOGLE_API_KEY") else
          "COMPLETA" if len(filas) == len(casos) else "INCOMPLETA")
informe = {"estado": estado, "fecha_utc": datetime.now(timezone.utc).isoformat(),
           "modelo": os.getenv("GEMINI_MODEL") or MODELO_PREDETERMINADO,
           "casos_ejecutados": len(filas), "casos_esperados": len(casos),
           "trayectorias_correctas": sum(f["trayectoria_ok"] for f in filas),
           "resultados_correctos": sum(f["resultado_ok"] for f in filas),
           "filas": filas}
Path("assets/evaluacion_actual.json").write_text(json.dumps(informe, indent=2, ensure_ascii=False), encoding="utf-8")
print("ESTADO ACTUAL:", estado)
if os.getenv("GOOGLE_API_KEY"):
    assert len(filas) == len(casos), "Evaluación interrumpida: no se ejecutaron todos los casos."
    assert all(f["trayectoria_ok"] and f["resultado_ok"] and
               f["confianza_transmitida_ok"] is not False for f in filas), "Hay fallos reales: revisar respuestas y trazas sin relajar el criterio."


Saludo | llamadas reales: [] | respuesta: ¡Hola! Bienvenido al lote Aliso. ¿En qué puedo ayudarte hoy?
Lectura | llamadas reales: [{'nombre': 'leer_digito', 'argumentos': {}}] | respuesta: Dígito: 2. Confianza: 57.95%
Inspección | llamadas reales: [{'nombre': 'leer_digito', 'argumentos': {}}, {'nombre': 'aplicar_regla', 'argumentos': {'confianza_pct': 57.95}}] | respuesta: Dígito: 2. Confianza: 57.95%. Decisión: REVISAR. Escalamiento: SEGUNDA_LECTURA (Motivo: Confianza baja: dos lecturas humanas independientes; resolver discrepancias.).
Memoria | llamadas reales: [] | respuesta: Me indicaste que tu lote se llama Aliso.
Casos con respuesta en vivo: 4 de 4
Trayectorias correctas: 4
Resultados correctos: 4
ESTADO ACTUAL: COMPLETA


C:\Users\wilso\Downloads\IA\taller-digitos\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\wilso\Downloads\IA\taller-digitos\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\wilso\Downloads\IA\taller-digitos\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\wilso\Downloads\IA\taller-digitos\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixe

,caso,esperadas,invocadas,trayectoria_ok,confianza_transmitida_ok,resultado_ok,respuesta
0,Saludo,[],[],True,None,True,¡Hola! Bienvenido al lote Aliso. ¿En qué puedo ayudarte hoy?
1,Lectura,[leer_digito],[leer_digito],True,None,True,Dígito: 2. Confianza: 57.95%
2,Inspección,"[leer_digito, aplicar_regla]","[leer_digito, aplicar_regla]",True,True,True,Dígito: 2. Confianza: 57.95%. Decisión: REVISAR. Escalamiento: SEGUNDA_LECTURA (Motivo: Confianza baja: dos lecturas humanas independientes; resolver discrepancias.).
3,Memoria,[],[],True,None,True,Me indicaste que tu lote se llama Aliso.
